# Macroeconomic Scenario Overlay

Applies three macroeconomic scenarios (Base, Adverse, Severe) to the ECL engine by scaling
PD according to assumed unemployment-linked stress multipliers, consistent with IFRS 9's
requirement to incorporate forward-looking macroeconomic information into ECL estimates.
Final "scenario-weighted ECL" blends all three scenarios using assigned probabilities,
representing a single forward-looking expected value rather than a single-point estimate.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/ecl_results.csv')
df.shape

(10000, 32)

In [2]:
scenario_multipliers = {
    'Base': 1.0,
    'Adverse': 1.5,
    'Severe': 2.5,
}

scenario_probabilities = {
    'Base': 0.60,
    'Adverse': 0.30,
    'Severe': 0.10,
}

In [3]:
for scenario, multiplier in scenario_multipliers.items():
    pd_col = f'PD_used_{scenario}'
    ecl_col = f'ECL_{scenario}'
    
    df[pd_col] = np.clip(df['PD_used'] * multiplier, 0, 1)
    df[ecl_col] = (df[pd_col] * df['LGD'] * df['EAD']).round(2)

df[['loan_id', 'stage', 'PD_used', 'ECL', 'PD_used_Base', 'ECL_Base', 'PD_used_Adverse', 'ECL_Adverse', 'PD_used_Severe', 'ECL_Severe']].head()

,loan_id,stage,PD_used,ECL,PD_used_Base,ECL_Base,PD_used_Adverse,ECL_Adverse,PD_used_Severe,ECL_Severe
0,1,Stage 2,0.590886,572.57,0.590886,572.57,0.886328,858.85,1.000000,969.00
1,2,Stage 1,0.150967,1084.72,0.150967,1084.72,0.226451,1627.08,0.377418,2711.80
2,3,Stage 1,0.022888,160.21,0.022888,160.21,0.034331,240.32,0.057219,400.53
3,4,Stage 1,0.054100,59.22,0.054100,59.22,0.081150,88.84,0.135250,148.06
4,5,Stage 1,0.103698,122.56,0.103698,122.56,0.155547,183.84,0.259244,306.41


In [4]:
df['ECL_weighted'] = (
    df['ECL_Base'] * scenario_probabilities['Base'] +
    df['ECL_Adverse'] * scenario_probabilities['Adverse'] +
    df['ECL_Severe'] * scenario_probabilities['Severe']
).round(2)

df[['loan_id', 'ECL_Base', 'ECL_Adverse', 'ECL_Severe', 'ECL_weighted']].head()

,loan_id,ECL_Base,ECL_Adverse,ECL_Severe,ECL_weighted
0,1,572.57,858.85,969.00,698.10
1,2,1084.72,1627.08,2711.80,1410.14
2,3,160.21,240.32,400.53,208.27
3,4,59.22,88.84,148.06,76.99
4,5,122.56,183.84,306.41,159.33


In [5]:
summary = pd.DataFrame({
    'Scenario': ['Base', 'Adverse', 'Severe', 'Weighted'],
    'Total ECL': [
        df['ECL_Base'].sum(),
        df['ECL_Adverse'].sum(),
        df['ECL_Severe'].sum(),
        df['ECL_weighted'].sum()
    ]
})
summary['Coverage Ratio'] = (summary['Total ECL'] / df['EAD'].sum()).round(4)
summary

,Scenario,Total ECL,Coverage Ratio
0,Base,7924494.99,0.0800
1,Adverse,11310569.21,0.1142
2,Severe,15525429.13,0.1567
3,Weighted,9700410.02,0.0979


In [6]:
df.to_csv('../data/processed/ecl_scenario_results.csv', index=False)
print("Saved.")

Saved.
